In [ ]:
# E11X — pulls the code from GitHub so a run is reproducible from a commit SHA
import os, subprocess, sys, shutil, time
REPO = "https://github.com/Ahmadrezanourozii/Automatic-detection-of-diabetic-retinopathy-and-grading-of-diabetic-macular-edema-using-CNN.git"
COMMIT = "53d0deb31d487b4d507c60b8fd22a585964f70cf"
WORK = "/kaggle/working/repo"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO, WORK], check=True)
if COMMIT and COMMIT != "HEAD":
    subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
sha = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"]).decode().strip()
print("CODE COMMIT", sha)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--pretty=%s"]).decode().strip())


In [ ]:
import os
for d in sorted(os.listdir("/kaggle/input")):
    n = sum(len(f) for _, _, f in os.walk(f"/kaggle/input/{d}"))
    print(f"{d:55s} {n:7d} files")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


In [ ]:
# The pool does not always honour the pinned accelerator. A P100 is sm_60 and the
# preinstalled torch cu128 build ships sm_70+ kernels only, so every CUDA call fails.
# Rather than lose the run, install a torch that supports this device (ISSUES.md §9).
import subprocess, sys, torch
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU {name}  sm_{cap[0]}{cap[1]}  torch {torch.__version__}")
ok = True
try:
    (torch.zeros(8, 8, device="cuda") @ torch.zeros(8, 8, device="cuda")).sum().item()
    print("kernels execute fine on this device")
except Exception as e:
    ok = False
    print("UNUSABLE:", e)
if not ok:
    print("installing a torch build that supports this GPU ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1", "torchvision==0.20.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)
    print("installed -- src/train.py runs in a subprocess so it picks up the new build")


In [ ]:
import subprocess, sys, os, glob, shutil
SRC = "ah22reza/dr-dme-e11"
OUT = f"/kaggle/working/E11X"
os.makedirs(OUT, exist_ok=True)
found = []
for root, _, files in os.walk("/kaggle/input"):
    for fn in files:
        if fn.startswith("best_") and fn.endswith(".pt"):
            found.append(os.path.join(root, fn))
        elif fn == "results.json" and "best_0.pt" in files:
            # The SOURCE run's results.json, kept for reference. It must NOT be written as
            # "results.json" -- that is this run's own reserved name, and copying another
            # run's file there is what made E08X look like it had produced E08's metrics
            # (ISSUES.md §20). Name it after where it came from.
            shutil.copy2(os.path.join(root, fn), f"{OUT}/source_run_results.json")
print(f"found {len(found)} fold weights from {SRC}")
for p_ in sorted(found):
    shutil.copy2(p_, os.path.join(OUT, os.path.basename(p_)))
if not found:
    raise SystemExit(f"no best_*.pt found under /kaggle/input -- is {SRC} attached "
                     f"as a kernel source?")
assert not os.path.exists(f"{OUT}/results.json"),     "results.json already exists in this run's output before the run produced one"

cmd = [sys.executable, "-u", "/kaggle/working/repo/src/eval_external.py",
       "--run", OUT, "--datasets", "/kaggle/input", "--corpus", "APTOS",
       "--cache", "/kaggle/temp/cache_ext", "--tta"]
print(" ".join(cmd), flush=True)
with open(f"{OUT}/external.log", "w") as f:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd="/kaggle/working/repo")
    for line in p.stdout:
        print(line, end="", flush=True); f.write(line); f.flush()
    p.wait()
print("exit=", p.returncode)
for p_ in glob.glob(f"{OUT}/best_*.pt"):
    os.remove(p_)          # they came from the source run; no need to duplicate them
